# Week 95, Day 1: Reproducing the issue

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w095d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('wraptext.py', 'w', encoding='utf-8') as f:
    f.write('"""Shorten text to fit a width.\n\n>>> shorten("Hello world, nice day", 18)\n\'Hello world, [...]\'\n"""\n\n\ndef shorten(text, width, placeholder=" [...]"):\n    """Collapse whitespace and cut text to fit width, ending with placeholder if anything was cut."""\n    words = text.split()\n    joined = " ".join(words)\n    if len(joined) <= width:\n        return joined\n    kept, length = [], 0\n    for word in words:\n        extra = len(word) + (1 if kept else 0)\n        if length + extra + len(placeholder) >= width:\n            break\n        kept.append(word)\n        length += extra\n    if not kept:\n        raise ValueError("placeholder too large for max width")\n    return " ".join(kept) + placeholder\n')
with open('history.py', 'w', encoding='utf-8') as f:
    f.write('GOOD = open("wraptext.py").read().replace("length + extra + len(placeholder) >= width", "length + extra + len(placeholder) > width")\nBAD = open("wraptext.py").read()\n\nCOMMITS = [\n    ("a1f3c20", "Add shorten()", GOOD),\n    ("b27e9d4", "Document shorten()", GOOD),\n    ("c3d81aa", "Collapse whitespace first", GOOD),\n    ("d4b0e6f", "Count the placeholder once", BAD),\n    ("e58c7b1", "Tidy names", BAD),\n    ("f6a2d93", "Add a module docstring", BAD),\n    ("g7e4410", "Raise ValueError for tiny widths", BAD),\n    ("h8c95fe", "Speed up the loop", BAD),\n]\n\n\ndef load(source):\n    namespace = {}\n    exec(source, namespace)\n    return namespace["shorten"]\n')

**Predict before you run:** what will this print?



In [ ]:
from wraptext import shorten

print(repr(shorten("Hello world, nice day", 18)))
print(len("Hello world, [...]"))

> **Key idea.** Decide it's a bug by the project's documentation, docstrings and tests; reproduce on current `main`. Shrink the input with an oracle (delta debugging) until nothing more can be removed. Bisect between a good and a bad commit (`git bisect`, or `git bisect run` with a script) to find where it started. Write up the version, minimal code, expected (and why), actual, and the first bad commit.

### Exercise: Running the reported cases

Write `check_cases(fn, cases)`: each case is `(args, expected)` with `args` a tuple. Call `fn(*args)` and return, for each case whose result differs, `"NAME(A, B): expected X, got Y"`, where NAME is `fn.__name__`, the arguments and results are written with `repr`, and arguments are separated by `", "`.

In [ ]:
def check_cases(fn, cases):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from wraptext import shorten\n\ndef test_cases():\n    \"the report, confirmed\"\n    cases = [((\"Hello world, nice day\", 18), \"Hello world, [...]\"), ((\"Hello world\", 40), \"Hello world\"),\n             ((\"Hello   world\", 11), \"Hello world\")]\n    assert check_cases(shorten, cases) == [\"shorten('Hello world, nice day', 18): expected 'Hello world, [...]', got 'Hello [...]'\"]\n    assert check_cases(len, [((\"abc\",), 3), (([1, 2],), 3)]) == [\"len([1, 2]): expected 3, got 2\"]")

### Exercise: What the documentation promises

Write `doc_failures(source)`: run the doctest examples in a module's source (its docstring and the docstrings of its functions and classes, found with `ast`) with `doctest`, in a namespace made by executing the source, and return the source of each failing example, stripped, in order.

In [ ]:
import ast
import doctest

def doc_failures(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_wraptext():\n    \"the docstring disagrees with the code\"\n    assert doc_failures(open(\"wraptext.py\").read()) == ['shorten(\"Hello world, nice day\", 18)']\n\ndef test_good():\n    \"examples that pass\"\n    src = '\"\"\"\\n>>> double(2)\\n4\\n\"\"\"\\n\\ndef double(x):\\n    \"\"\"\\n    >>> double(5)\\n    10\\n    >>> double(0)\\n    1\\n    \"\"\"\\n    return 2 * x\\n'\n    assert doc_failures(src) == [\"double(0)\"]")

### Exercise: The smallest failing input

Write `minimise(words, still_fails)`: start from the list of words; repeatedly try removing one word at a time (from the first), and as soon as a non-empty smaller list still fails (by `still_fails(candidate)`), keep it and start again from the first word. Stop when no single removal still fails, and return the list.

In [ ]:
def minimise(words, still_fails):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from history import COMMITS, load\n\nGOOD, BAD = load(COMMITS[0][2]), load(COMMITS[-1][2])\n\ndef test_shorten():\n    \"a long report made small\"\n    text = \"Hello   world, what a lovely nice day today\"\n    fails = lambda ws: BAD(\" \".join(ws), 18) != GOOD(\" \".join(ws), 18)\n    assert fails(text.split())\n    assert minimise(text.split(), fails) == [\"Hello\", \"a\", \"nice\", \"day\", \"today\"]\n\ndef test_generic():\n    \"any oracle\"\n    assert minimise(list(\"abcdef\"), lambda ws: \"c\" in ws and \"e\" in ws) == [\"c\", \"e\"]\n    assert minimise([\"x\"], lambda ws: True) == [\"x\"]")

### Exercise: Bisecting the history

Write `bisect_commits(commits, is_bad)`: `commits` is a list from oldest to newest, the first known good and the last known bad. Keep a range `lo` (good) to `hi` (bad); while they're more than one apart, test the commit halfway (`(lo + hi) // 2`) and move one end to it. Return `(ID, calls)`: the first bad commit's id (each commit is `(id, message, source)`) and how many times you called `is_bad`.

In [ ]:
def bisect_commits(commits, is_bad):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from history import COMMITS, load\n\nGOOD = load(COMMITS[0][2])\n\ndef test_history():\n    \"the commit that introduced it\"\n    is_bad = lambda c: load(c[2])(\"Hello world, nice day\", 18) != GOOD(\"Hello world, nice day\", 18)\n    assert bisect_commits(COMMITS, is_bad) == (\"d4b0e6f\", 3)\n\ndef test_long():\n    \"a thousand commits, ten tests\"\n    commits = [(f\"c{i}\", \"\", None) for i in range(1000)]\n    assert bisect_commits(commits, lambda c: int(c[0][1:]) >= 637) == (\"c637\", 10)")

### Exercise: Writing the reproduction

Write `repro_comment(version, code, expected, actual, why, commit=None)`, returning Markdown lines joined by newlines: `Reproduced on VERSION:`, a blank line, a fenced Python block (three backticks and `python`, the code, three backticks), a blank line, `Expected: EXPECTED (WHY)`, `Actual: ACTUAL`, and, if `commit` is given, a blank line and `First bad commit: COMMIT`.

In [ ]:
def repro_comment(version, code, expected, actual, why, commit=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("FENCE = \"`\" * 3\n\ndef test_comment():\n    \"readable in ten seconds\"\n    text = repro_comment(\"main (3.15.0a1+, 2ebdaf8)\", \">>> shorten('Hello a nice day today', 18)\\n'Hello a [...]'\",\n                         \"'Hello a nice [...]'\", \"'Hello a [...]'\", \"it is exactly 18 characters\", \"d4b0e6f Count the placeholder once\")\n    assert text == (f\"Reproduced on main (3.15.0a1+, 2ebdaf8):\\n\\n{FENCE}python\\n>>> shorten('Hello a nice day today', 18)\\n'Hello a [...]'\\n{FENCE}\\n\\n\"\n                    \"Expected: 'Hello a nice [...]' (it is exactly 18 characters)\\nActual: 'Hello a [...]'\\n\\nFirst bad commit: d4b0e6f Count the placeholder once\")\n    assert repro_comment(\"3.14.0\", \"x\", \"1\", \"2\", \"why\").endswith(\"Actual: 2\")")

### Bug hunt: The last good commit, not the first bad one

At the end of a bisection, `lo` is the last good commit and `hi` the first bad one, but this returns `lo` and prints `c3d81aa`. Return the first bad commit, so it prints `d4b0e6f`.

In [ ]:
ids = ["a1f3c20", "b27e9d4", "c3d81aa", "d4b0e6f", "e58c7b1", "f6a2d93", "g7e4410", "h8c95fe"]
bad_from = 3

lo, hi = 0, len(ids) - 1
while hi - lo > 1:
    mid = (lo + hi) // 2
    if mid >= bad_from:
        hi = mid
    else:
        lo = mid
print(ids[lo])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"hi is bad\"\n    expect_output(\"d4b0e6f\")")

### Bug hunt: Every case fails

`is not` compares identity, and strings built at run time are rarely the same object as a literal, so this reports both cases as failing and prints `2`. Compare values so it prints `1`.

In [ ]:
from wraptext import shorten

cases = [(("Hello world, nice day", 18), "Hello world, [...]"), (("Hello  there", 40), "Hello there")]
print(sum(1 for args, expected in cases if shorten(*args) is not expected))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"equality, not identity\"\n    expect_output(\"1\")")

### Puzzle: Halving the range

Arrange the bisection loop: test the middle commit and move the right end of the range. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # len("Hello world, [...]")
q = ___   # how many tests bisection needs for about 1000 commits (a whole number)
r = ___   # the git command that automates bisection with a script (three words)
s = ___   # True or False: reproduce on an old release rather than current main
t = ___   # the exit status a bisect script uses for a good commit

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 18 and q == 10 and r == \"git bisect run\", \"p, q and r\"\n    assert s is False and t == 0, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a reproduction report

Write `reproduction_report(text, width, commits)` using today's helpers (given) and `load` from `history.py`. The first commit is the oracle (known good), the last is current. Return a list of lines:

1. `"actual: A (expected E)"` with both written with `repr`, or `"no bug: R"` (and nothing else) if they agree;
2. `"minimal: shorten(T, WIDTH)"` (T with `repr`) from minimising the words of `text` with width fixed, using "the current result differs from the oracle's" as the test;
3. `"first bad commit: ID MESSAGE (N tests)"` from bisecting with the minimal text;
4. the lines of `repro_comment("main", CODE, E, A, "the oracle's answer", "ID MESSAGE")`, where CODE is `>>> shorten(T, WIDTH)` and the current result's `repr` on the next line, and E and A are for the minimal text.

In [ ]:
from history import load

def minimise(words, still_fails):
    words = list(words)
    changed = True
    while changed:
        changed = False
        for i in range(len(words)):
            candidate = words[:i] + words[i + 1:]
            if candidate and still_fails(candidate):
                words, changed = candidate, True
                break
    return words

def bisect_commits(commits, is_bad):
    lo, hi, calls = 0, len(commits) - 1, 0
    while hi - lo > 1:
        mid = (lo + hi) // 2
        calls += 1
        if is_bad(commits[mid]):
            hi = mid
        else:
            lo = mid
    return commits[hi][0], calls

def repro_comment(version, code, expected, actual, why, commit=None):
    fence = "`" * 3
    lines = [f"Reproduced on {version}:", "", f"{fence}python", code, fence, "", f"Expected: {expected} ({why})", f"Actual: {actual}"]
    if commit:
        lines += ["", f"First bad commit: {commit}"]
    return "\n".join(lines)

def reproduction_report(text, width, commits):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from history import COMMITS\n\ndef test_report():\n    \"from a long report to a clear comment\"\n    lines = reproduction_report(\"Hello   world, what a lovely nice day today\", 18, COMMITS)\n    assert lines[:3] == [\"actual: 'Hello [...]' (expected 'Hello world, [...]')\",\n                         \"minimal: shorten('Hello a nice day today', 18)\",\n                         \"first bad commit: d4b0e6f Count the placeholder once (3 tests)\"]\n    assert lines[3:] == [\"Reproduced on main:\", \"\", \"`\" * 3 + \"python\", \">>> shorten('Hello a nice day today', 18)\", \"'Hello a [...]'\", \"`\" * 3, \"\",\n                         \"Expected: 'Hello a nice [...]' (the oracle's answer)\", \"Actual: 'Hello a [...]'\", \"\",\n                         \"First bad commit: d4b0e6f Count the placeholder once\"]\n\ndef test_no_bug():\n    \"nothing to report\"\n    assert reproduction_report(\"Hello there\", 40, COMMITS) == [\"no bug: 'Hello there'\"]")

## Warm-up

`"Hello world, [...]"` is exactly 18 characters, so it fits, yet `shorten` stopped after `Hello`. That's the reported bug, reproduced. Notice what made it quick: the exact call, the exact output, and the length that proves the expected answer fits.

## Is it really a bug?

Before anything else, decide whether the behaviour is wrong, by the project's own standards: its **documentation**, its **docstrings** and examples, and its **tests**. Here the module's own docstring says `shorten("Hello world, nice day", 18)` gives `'Hello world, [...]'`, so the code disagrees with its documentation, and doctest can show it. If the documentation were silent, the issue would need a maintainer's decision first: some "bugs" are intended behaviour, and some are changes in behaviour that people now rely on.

Reproduce on the **current `main`** (your own build, last week), not an old release: the bug may already be fixed.

## Making it small

A long report ("I shortened my 400-word essay and a word went missing") is hard to reason about. Shrink it: remove a piece of the input, check the bug still happens, keep the smaller version, and repeat until nothing more can go. This is the idea behind **delta debugging** (Andreas Zeller's work, which also gives its name to the general technique). You need an **oracle** to decide "still fails": here, the result differs from a version known to be correct (the first commit), or from the answer you can prove by hand.

## When did it start?

If the bug wasn't always there, the commit that introduced it explains it. `git bisect` finds that commit by **binary search**: mark one commit good and a later one bad, and Git checks out the commit halfway; you test it and say `git bisect good` or `git bisect bad`, and the range halves each time. A thousand commits take about ten tests. `git bisect run ./check.sh` automates it with a script that exits 0 for good and 1 for bad. `history.py` holds `wraptext`'s eight commits so you can bisect them here.

## Writing it up

A reproduction a maintainer can trust has: the **version** (and commit, if you built it), the **smallest code** that shows the problem, the **expected** result with why, the **actual** result, and, if you found it, the **commit** that introduced it. Put the code in a block that can be pasted and run as it is. That comment is often the most useful thing anyone writes on an issue.

## Practice

## Quiz

## Daily challenge

## Your turn

1. **Reproduce your issue** (the one you chose last week) on your build of `main`, with the exact command and output.
2. **Shrink it** until nothing more can be removed, by hand or with a script.
3. **Bisect it** if it's a regression: `git bisect start`, `git bisect bad`, `git bisect good v3.13.0`, then test each commit Git offers (or `git bisect run`). Finish with `git bisect reset`.
4. **Write the reproduction** as a comment on the issue, if it adds something the issue doesn't already have.

## Before you go

Tomorrow you'll write the fix: understand why the code goes wrong, then make the smallest change that puts it right without changing anything else.